# 02 · Pipeline walkthrough (demo notebook)

Runs the whole MLOps loop from one place — handy for rehearsing the live demo.
Each step is a script in `src/`, so CI and teammates run *exactly* the same code.

In [1]:
import os
from pathlib import Path

while not (Path.cwd() / "src").is_dir() and Path.cwd() != Path.cwd().parent:
    os.chdir("..")
Path.cwd()

PosixPath('/home/jovyan')

In [2]:
!python -m src.data --kaggle

UCI Wine Quality (red + white Vinho Verde)
  winequality-red.csv already present
  winequality-white.csv already present


  6497 rows, 1177 exact duplicates removed -> 5320
  tiers: {'good': 2323, 'standard': 1988, 'premium': 1009}
  -> /home/jovyan/data/processed/wine_quality.csv (sha aa35c5597a8d)

Kaggle Wine Reviews -> Portuguese catalog
  winemag-data-130k-v2.csv already present


  5256 Portuguese wines | colours: {'red': 3302, 'white': 1481, 'rosé': 239, 'unknown': 234}
  -> /home/jovyan/data/processed/portugal_wines.csv


In [3]:
# Train all configurations -> MLflow http://localhost:5001 (experiment VinhoVerde-Quality)
!python -m src.train

Training 4 configurations on 5320 rows (data sha aa35c5597a8d)



  logreg_baseline  macro-F1 0.543 | acc 0.547 | premium recall 0.738
🏃 View run logreg_baseline at: http://mlflow:5000/#/experiments/1/runs/0c7b96019e0545ba876a324a725312d0
🧪 View experiment at: http://mlflow:5000/#/experiments/1


  rf_balanced      macro-F1 0.599 | acc 0.601 | premium recall 0.649


🏃 View run rf_balanced at: http://mlflow:5000/#/experiments/1/runs/059ab6c4665f419ba7584568e9538790
🧪 View experiment at: http://mlflow:5000/#/experiments/1


  HistGB           macro-F1 0.563 | acc 0.572 | premium recall 0.465


🏃 View run HistGB at: http://mlflow:5000/#/experiments/1/runs/7852926f8a9e4f4da6930f802c60bfe7
🧪 View experiment at: http://mlflow:5000/#/experiments/1


  rf_unbalanced    macro-F1 0.595 | acc 0.611 | premium recall 0.446


🏃 View run rf_unbalanced at: http://mlflow:5000/#/experiments/1/runs/167a6774c10f466bb183d01013687698
🧪 View experiment at: http://mlflow:5000/#/experiments/1

Best: rf_balanced (macro-F1 0.599). Next: python -m src.register


In [4]:
# Register the best run -> @candidate -> gate -> @champion
!python -m src.register

Best run: rf_balanced (059ab6c4) macro-F1 0.599


Registered model 'VinhoVerde-Quality-Classifier' already exists. Creating a new version of this model...


2026/09/30 07:24:19 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: VinhoVerde-Quality-Classifier, version 5
Created version '5' of model 'VinhoVerde-Quality-Classifier'.


Registered VinhoVerde-Quality-Classifier v5 -> @candidate


Current champion v4 macro-F1 0.599

Candidate v5 macro-F1 0.599
  [PASS] macro_f1 >= 0.55
  [PASS] not worse than champion



@champion -> v5. Tell the API:  curl -X POST http://localhost:8080/reload


In [5]:
import requests
wine = requests.get("http://app:8080/examples").json()["red"]
requests.post("http://app:8080/reload").json(), requests.post("http://app:8080/predict", json=wine).json()

({'reloaded': True, 'model_version': '5'},
 {'quality_tier': 'standard',
  'probabilities': {'good': 0.184, 'premium': 0.028, 'standard': 0.789},
  'action': 'Blend or sell as house wine / bulk. Do not bottle under the estate label.',
  'model_version': '5',
  'input': {'fixed acidity': 7.4,
   'volatile acidity': 0.7,
   'citric acid': 0.0,
   'residual sugar': 1.9,
   'chlorides': 0.076,
   'free sulfur dioxide': 11.0,
   'total sulfur dioxide': 34.0,
   'density': 0.9978,
   'pH': 3.51,
   'sulphates': 0.56,
   'alcohol': 9.4,
   'wine_type': 'red'}})

## LLMOps: prompt versions
~6 minutes on the free tier (5 requests/minute).

In [6]:
!python -m src.evaluate_prompts --promote

Evaluating 3 modes x 13 cases with gemini-3.5-flash-lite (~8 min)



2026/09/30 07:24:24 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for prompt version to finish creation. Prompt name: sommelier, version 13


HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"


HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"


HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"


...

🏃 View run grounded_structured at: http://mlflow:5000/#/experiments/2/runs/84e9a1e5a5c14056a872c0d04f84bfa5
🧪 View experiment at: http://mlflow:5000/#/experiments/2


  grounded_structured  overall 0.883 | coverage 0.67 | grounding 1.00 | refusals 1.00 | false refusals 0.00



Gate passed for grounded.


@champion -> sommelier v14. Reload: curl -X POST http://localhost:8080/prompt/reload


🏃 View run prompt-mode-comparison at: http://mlflow:5000/#/experiments/2/runs/998a82e339054e17baea2ae2182df105
🧪 View experiment at: http://mlflow:5000/#/experiments/2
2026/09/30 07:32:01 INFO mlflow.tracing.export.async_export_queue: Flushing the async trace logging queue before program exit. This may take a while...


In [7]:
requests.post("http://app:8080/prompt/reload").json()

{'reloaded': True, 'prompt_version': 14, 'mode': 'grounded'}

In [8]:
r = requests.post("http://app:8080/chat", json={"message": "A Douro red under 20 euros for steak?"}).json()
print(r["prompt_mode"], r["prompt_version"]); print(r["answer"])

grounded 14
Olá! As a Portuguese sommelier, I would be delighted to help you find a fantastic Douro red under €20 to pair with a good steak. 

Steak calls for a wine with enough structure, tannins, and fruit intensity to cut through the richness of the meat. From our shop catalog, I have a few wonderful options for you:

1. **Quinta da Pacheca 2013 Superior Tinto Red (Douro)** (€14, 88 pts) – This is a great choice. With generous tannins and berry flavors, it has the structure to stand up nicely to a juicy steak.
2. **Conceito Vinhos 2014 Contraste Tinto Red (Douro)** (€19, 87 pts) – If you prefer something with dark tannins and good concentration, along with spice and wood-aging flavors, this will make a robust and satisfying pairing.
3. **Quevedo 2014 Oscar's Red (Douro)** (€15, 86 pts) – An accessible wine full of red berries, bright acidity, and a supportive layer of tannins.
4. **Caves Velhas 2015 Encostas do Douro Red (Douro)** (€9, 85 pts) – A more budget-friendly option that is

## Demo trick: show a promotion live
1. Add a new configuration to `CONFIGS` in `src/train.py` (e.g. more trees).
2. `!python -m src.train --only <new_name>` → `!python -m src.register`
3. Gate passes or blocks it → `/reload` → the UI footer shows the new model version.